# Squat-Only YOLO Pose Extraction (Google Colab)

This notebook keeps the local pipeline unchanged and provides a separate Colab workflow for running squat-only pose extraction with a GPU.

## Reason, Approach, Idea, Result Interpretation

**Reason for this section**
- This stage converts raw squat videos into pose sequences that later notebooks can reuse.
- It is separated from later stages so YOLO extraction only needs to run once.

**Approach**
- Build a squat-only pose index.
- Run a pretrained YOLO pose model on every video frame.
- Save one `.npy` pose sequence per video with shape `[T, 51]`.

**Core idea**
- We are not training YOLO here.
- We are using a pretrained pose model to detect human keypoints for downstream motion analysis.

**How to interpret results**
- `ok` means pose extraction succeeded and a pose file was written.
- `skipped_exists` means an existing output was reused.
- `failed` means the video path or extraction step needs investigation.
- `ok_with_zero_pose_frames` should stay very low in a healthy run.


## 1. Select a GPU Runtime

In Colab:
- `Runtime` -> `Change runtime type`
- `Hardware accelerator` -> `GPU`

**Why this section exists**
YOLO pose inference is the most expensive part of this pipeline, so runtime selection directly affects throughput and feasibility in Colab.

**Approach / idea**
Use a GPU-backed Colab session before installing dependencies or launching extraction so the same runtime is used end to end.

**Result interpretation**
If the runtime is still CPU-only, extraction will still work but it will be much slower and the later timing expectations will no longer apply.


In [ ]:
import os

REPO_URL = "https://github.com/lindaperez/CV_Image_pose_detection.git"
CODE_ROOT = "/content/CV_Image_pose_detection"
WORKDIR = CODE_ROOT

# Persist data artifacts in Drive so they survive Colab runtime resets.
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection"
DRIVE_VIDEO_DIR = f"{DRIVE_PROJECT_ROOT}/Data/LLSP/video"
DRIVE_ANNOTATION_DIR = f"{DRIVE_PROJECT_ROOT}/Data/LLSP/annotation_cleaned"
DRIVE_POSE_FEATURE_DIR = f"{DRIVE_ANNOTATION_DIR}/pose_features"

print('CODE_ROOT =', CODE_ROOT)
print('DRIVE_VIDEO_DIR =', DRIVE_VIDEO_DIR)
print('DRIVE_ANNOTATION_DIR =', DRIVE_ANNOTATION_DIR)
print('DRIVE_POSE_FEATURE_DIR =', DRIVE_POSE_FEATURE_DIR)

CODE_ROOT = /content/CV_Image_pose_detection
DRIVE_VIDEO_DIR = /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/video
DRIVE_ANNOTATION_DIR = /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/annotation_cleaned
DRIVE_POSE_FEATURE_DIR = /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_features


## 2. Get the Repo into Colab

**Why this section exists**
The extraction notebook depends on project scripts, cleaned annotations, and the same folder structure used locally.

**Approach / idea**
Bring the repo into `/content` and keep data paths explicit so code in the notebook and code in the repo stay aligned.

**Result interpretation**
If this section completes cleanly, later failures are less likely to be missing-script or wrong-working-directory issues.


In [ ]:
# Run this cell if you want to clone from GitHub.
!rm -r /content/CV_Image_pose_detection
!git clone $REPO_URL /content/CV_Image_pose_detection

rm: cannot remove '/content/CV_Image_pose_detection': No such file or directory
Cloning into '/content/CV_Image_pose_detection'...
remote: Enumerating objects: 219, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 219 (delta 6), reused 13 (delta 5), pack-reused 203 (from 1)
Receiving objects: 100% (219/219), 440.91 MiB | 9.98 MiB/s, done.
Resolving deltas: 100% (71/71), done.
Updating files: 100% (141/141), done.


In [ ]:
%cd $WORKDIR
!pwd
!ls /content/CV_Image_pose_detection/artifacts/3_Modeling

/content/CV_Image_pose_detection
/content/CV_Image_pose_detection
3_Model_Training_01.ipynb		__pycache__
4_Squat_Pose_Extraction_Colab.ipynb	training_outputs
5_Squat_Feature_Extraction_Colab.ipynb	yolo11n-pose.pt
build_pose_feature_index.py		YOLO_PIPELINE.md
COLAB_SQUAT_POSE.md			YOLO_POSE_STAGE.md
pose_feature_extraction.py


In [ ]:
cd /content/

/content


## 3. Install Pose Dependencies

**Why this section exists**
Pose extraction depends on `ultralytics`, `opencv-python`, and `numpy`, which are not guaranteed to be present in a fresh Colab runtime.

**Approach / idea**
Install the dedicated pose requirements file once in the active runtime before loading the YOLO model.

**Result interpretation**
Successful installation means later errors are more likely to be dataset or path problems rather than missing packages.


In [ ]:
!python3 -m pip install -r CV_Image_pose_detection/requirements-pose.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 72.0 MB/s eta 0:00:00


## 4. Confirm GPU

**Why this section exists**
It is easy to request a GPU runtime in Colab but still end up using CPU if CUDA is unavailable or misconfigured.

**Approach / idea**
Run a direct GPU check before extraction so the device decision is explicit rather than assumed.

**Result interpretation**
If CUDA is visible here, the extraction stage should use the requested GPU device; if not, revisit runtime settings before spending time on full runs.


In [ ]:
import torch

print("cuda_available =", torch.cuda.is_available())
print("device_count =", torch.cuda.device_count())
print("device_name =", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

cuda_available = True
device_count = 1
device_name = Tesla T4


In [19]:
#connect with drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 5. Build a Squat-Only Index

**Why this section exists**
The full dataset is larger than needed for this end-to-end prototype, so this section narrows the workload to the squat subset only.

**Approach / idea**
Generate `pose_feature_index_squat.csv` from the cleaned annotations and point output feature paths at the persistent Drive location.

**Result interpretation**
The row count here is the planned workload for pose extraction; if it looks wrong, fix the index before running YOLO.


In [ ]:
!python3 CV_Image_pose_detection/artifacts/3_Modeling/build_pose_feature_index.py \
  --exercise squat \
  --feature-dir $DRIVE_POSE_FEATURE_DIR \
  --output-csv $DRIVE_ANNOTATION_DIR/pose_feature_index_squat.csv

Wrote 118 rows for squat to /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_feature_index_squat.csv


In [ ]:
ls /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/

pose_extraction_report.csv    pose_feature_index_squat.csv
pose_extraction_summary.json  pose_features/


In [ ]:
ls -l CV_Image_pose_detection/Data/LLSP/annotation_cleaned/

total 576
drwxr-xr-x 2 root root   4096 Mar  8 22:20 _archived_test_outputs_20260304_114631/
-rw-r--r-- 1 root root    178 Mar  8 22:20 class_weights_train.csv
-rw-r--r-- 1 root root   3654 Mar  8 22:20 decisions_manifest.json
-rw-r--r-- 1 root root   6230 Mar  8 22:20 pose_extraction_report.csv
-rw-r--r-- 1 root root    533 Mar  8 22:20 pose_extraction_summary.json
-rw-r--r-- 1 root root 170393 Mar  8 22:20 pose_feature_index.csv
-rw-r--r-- 1 root root  21266 Mar  8 22:20 pose_feature_index_squat.csv
drwxr-xr-x 2 root root   4096 Mar  8 22:20 pose_features/
-rw-r--r-- 1 root root 281824 Mar  8 22:20 train_cleaned.csv
-rw-r--r-- 1 root root  26008 Mar  8 22:20 train_sample_weights.csv
-rw-r--r-- 1 root root  51740 Mar  8 22:20 valid_cleaned.csv


In [ ]:
!head -n 5 $DRIVE_ANNOTATION_DIR/pose_feature_index_squat.csv

name,feature_path,type,split,count
test2340.mp4,/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_features/test2340.npy,squat,train,4.0
stu4_66.mp4,/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_features/stu4_66.npy,squat,train,27.0
stu9_63.mp4,/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_features/stu9_63.npy,squat,train,20.0
stu1_68.mp4,/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_features/stu1_68.npy,squat,train,20.0


In [ ]:
# Check on number of videos in the directory

from pathlib import Path

def count_mp4_videos(directory: str, recursive: bool = True) -> int:
    """
    Count .mp4 video files in a directory.

    Args:
        directory: Path to the folder.
        recursive: If True, also count videos in subfolders.

    Returns:
        Number of .mp4 files found.
    """
    path = Path(directory)

    if not path.exists():
        raise FileNotFoundError(f"Directory does not exist: {directory}")

    if not path.is_dir():
        raise NotADirectoryError(f"Not a directory: {directory}")

    pattern = "**/*.mp4" if recursive else "*.mp4"
    return sum(1 for _ in path.glob(pattern))


drive_dir = "/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/video/"
total_videos = count_mp4_videos(drive_dir)
print(f"Total .mp4 videos: {total_videos}")



Total .mp4 videos: 1041


## 6. Smoke Test on 5 Squat Videos

**Why this section exists**
A small smoke test catches path, dependency, and model-loading issues before committing GPU time to the full batch.

**Approach / idea**
Run the same extraction pipeline on a very small subset using the exact same inputs and output layout as the full run.

**Result interpretation**
If the smoke test succeeds, the full run is mostly a scale-up problem; if it fails, fix the issue here first.


In [ ]:
  !python3 /content/CV_Image_pose_detection/artifacts/3_Modeling/pose_feature_extraction.py \
  --index-csv $DRIVE_ANNOTATION_DIR/pose_feature_index_squat.csv \
  --video-dir $DRIVE_VIDEO_DIR \
  --model /content/CV_Image_pose_detection/artifacts/3_Modeling/yolo11n-pose.pt \
  --report-path $DRIVE_ANNOTATION_DIR/pose_extraction_report.csv \
  --summary-path $DRIVE_ANNOTATION_DIR/pose_extraction_summary.json \
  --device cuda:0 \
  --max-videos 5 \
  --overwrite

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Indexed 1041 videos under /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/video
Processing 5 videos...
[5/5] ok: stu6_62.mp4 | frames=749 used=749 shape=(749, 51)

Done.
ok=5, skipped_exists=0, failed=0
report: /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_extraction_report.csv
summary: /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_extraction_summary.json


## 7. Run the Full Squat Set

Remove `--overwrite` if you want to resume and skip already-finished files.

**Why this section exists**
This is the production pass that creates the persistent pose features used by the downstream squat pipeline.

**Approach / idea**
Reuse the same extractor and index from the smoke test, but remove the small-sample restriction, force regeneration with `--overwrite`, and write outputs to Drive.

**Result interpretation**
The final summary here is the authoritative extraction status: `ok`, `skipped_exists`, and `failed` tell you whether the pose stage is complete.


In [ ]:
!python3 CV_Image_pose_detection/artifacts/3_Modeling/pose_feature_extraction.py \
  --index-csv $DRIVE_ANNOTATION_DIR/pose_feature_index_squat.csv \
  --video-dir $DRIVE_VIDEO_DIR \
  --model /content/CV_Image_pose_detection/artifacts/3_Modeling/yolo11n-pose.pt \
  --report-path $DRIVE_ANNOTATION_DIR/pose_extraction_report.csv \
  --summary-path $DRIVE_ANNOTATION_DIR/pose_extraction_summary.json \
  --device cuda:0 \
  --overwrite

Indexed 1041 videos under /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/video
Processing 118 videos...
[25/118] ok: stu5_66.mp4 | frames=990 used=990 shape=(990, 51)
[50/118] ok: stu7_69.mp4 | frames=1109 used=1109 shape=(1109, 51)
[75/118] ok: stu1_67.mp4 | frames=1079 used=1079 shape=(1079, 51)
[100/118] ok: stu9_66.mp4 | frames=1800 used=1800 shape=(1800, 51)
[118/118] ok: stu10_69.mp4 | frames=1920 used=1920 shape=(1920, 51)

Done.
ok=113, skipped_exists=5, failed=0
report: /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_extraction_report.csv
summary: /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_extraction_summary.json


## 8. Inspect Outputs

**Why this section exists**
Before moving to feature engineering, verify that the expected pose files, report CSV, and summary JSON were actually written.

**Approach / idea**
Check output counts and inspect a few artifacts directly rather than assuming a successful log means complete usable output.

**Result interpretation**
If the outputs look correct here, step 5 can treat pose extraction as finished and read these files as the next-stage contract.


In [ ]:
import json
from pathlib import Path

summary_path = Path("CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_extraction_summary.json")
print(json.loads(summary_path.read_text()))

{'total_rows': 20, 'ok': 20, 'skipped_exists': 0, 'failed': 0, 'ok_with_zero_pose_frames': 0, 'args': {'index_csv': '/Users/lindaperez/Documents/COMPUTER_VISION/Final_project/personal-git/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_feature_index.csv', 'video_dir': '/Users/lindaperez/Documents/COMPUTER_VISION/Final_project/personal-git/CV_Image_pose_detection/Data/LLSP/video', 'model': 'yolo11n-pose.pt', 'conf': 0.25, 'imgsz': 640, 'overwrite': False, 'max_videos': 20}}


In [ ]:
import pandas as pd

report = pd.read_csv(f"{DRIVE_ANNOTATION_DIR}/pose_extraction_report.csv")
report.head()

,name,video_path,feature_path,status,frames_total,frames_used,feat_dim,message
0,test2340.mp4,/content/drive/MyDrive/FinalProjectCV/CV_Image...,/content/drive/MyDrive/FinalProjectCV/CV_Image...,skipped_exists,0,0,0,feature file already exists
1,stu4_66.mp4,/content/drive/MyDrive/FinalProjectCV/CV_Image...,/content/drive/MyDrive/FinalProjectCV/CV_Image...,skipped_exists,0,0,0,feature file already exists
2,stu9_63.mp4,/content/drive/MyDrive/FinalProjectCV/CV_Image...,/content/drive/MyDrive/FinalProjectCV/CV_Image...,skipped_exists,0,0,0,feature file already exists
3,stu1_68.mp4,/content/drive/MyDrive/FinalProjectCV/CV_Image...,/content/drive/MyDrive/FinalProjectCV/CV_Image...,skipped_exists,0,0,0,feature file already exists
4,stu6_62.mp4,/content/drive/MyDrive/FinalProjectCV/CV_Image...,/content/drive/MyDrive/FinalProjectCV/CV_Image...,skipped_exists,0,0,0,feature file already exists


# What it means:

total_rows: 118
The squat index contained 118 videos.

ok: 113

113 videos were processed and pose features were written in this run.

skipped_exists: 5

5 feature files already existed, so the script skipped them.

failed: 0

No missing videos and no runtime failures.

ok_with_zero_pose_frames: 0

YOLO found at least some pose frames in every successful video.



* 118 squat videos
* -> YOLO pose extraction completed
* -> pose feature files available
* -> ready for post-processing / feature engineering
'''